In [142]:
import requests
import json
from datetime import datetime
from datetime import timezone
import time
import random
import asyncio, aiohttp

_TOKEN_CACHE = None  # global cache

def get_session_token():
    global _TOKEN_CACHE
    if _TOKEN_CACHE:   # reuse valid token
        return _TOKEN_CACHE

    url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    handle = "repostproj.bsky.social"
    app_password = "vyvc-xg5q-seda-utaz" 

    for _ in range(3):  # retry a few times on rate limit
        r = requests.post(url, json={"identifier": handle, "password": app_password})
        if r.status_code == 429:
            print("⚠️ Rate-limited on login, sleeping 5s...")
            time.sleep(1)
            continue
        r.raise_for_status()
        _TOKEN_CACHE = r.json()["accessJwt"]
        return _TOKEN_CACHE

    raise RuntimeError("Failed to obtain token after retries")


In [143]:
file = "test_30000_trump.jsonl"

In [141]:


def search_bluesky_posts(
    query,
    limit=10000,
    outfile=file,
    since=None,
    until=None,
    lang="en",
    sort="latest",
):
    """
    Search Bluesky posts using the authenticated app.bsky.feed.searchPosts endpoint.
    Supports since/until, lang, sort.
    """
    url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    token = get_session_token()
    headers = {
        "Authorization": f"Bearer {token}",
        "User-Agent": "Mozilla/5.0",
        "Accept-Encoding": "gzip",
    }

    cursor, total = None, 0

    def fmt(dt):
        if dt is None:
            return None
        if isinstance(dt, datetime):
            # Use Zulu (UTC) format that Bluesky expects
            return dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        return str(dt)

    since = fmt(since)
    until = fmt(until)

    with open(outfile, "a", encoding="utf-8") as f:
        while total < limit:
            params = {
                "q": query,
                "sort": sort,
                "lang": lang,
                "limit": min(100, limit - total),
            }
            if cursor:
                params["cursor"] = cursor
            if since:
                params["since"] = since
            if until:
                params["until"] = until

            r = requests.get(url, params=params, headers=headers)
            if r.status_code in (429, 502):
                print(f"⚠️ Rate limited ({r.status_code}) — waiting 1s...")
                time.sleep(1)
                continue
            if r.status_code == 401:
                raise RuntimeError("Invalid or expired token — get a new session token.")
            if r.status_code != 200:
                raise RuntimeError(f"Error {r.status_code}: {r.text}")

            data = r.json()
            posts = data.get("posts", [])
            if not posts:
                print("ℹNo posts in this batch — stopping.")
                break

            for p in posts:
                json.dump(p, f, ensure_ascii=False)
                f.write("\n")
                total += 1
                if total >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                print("ℹNo more cursor — reached end of results.")
                break
            if round(total / 1000) != round((total-100) / 1000) and total >= 1000:
                print(f"🧾 Collected {total} posts so far...")

    print(f"✅ Saved {total} posts to {outfile}")


In [140]:




search_bluesky_posts(
    query="trump",
    limit=5000,
    outfile=file,
    until=datetime(2025, 10, 9),
    since=datetime(2025,9,1)
)

🧾 Collected 596 posts so far...
🧾 Collected 1586 posts so far...
🧾 Collected 2574 posts so far...
🧾 Collected 3556 posts so far...
🧾 Collected 4542 posts so far...
✅ Saved 5000 posts to test_30000_trump.jsonl


In [144]:




def count_trump_tagged_posts(tag, file_path=file):
    """
    Counts how many Bluesky posts in a JSONL file include 'trump' as a tag.
    
    Args:
        file_path (str): Path to the JSONL file containing Bluesky post objects.
    
    Returns:
        int: Number of posts that have 'trump' as a tag.
    """
    count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                post = json.loads(line.strip())
                record = post.get('record', {})
                facets = record.get('facets', [])
                
                # Extract tags from facets
                tags = []
                for facet in facets:
                    for feature in facet.get('features', []):
                        if feature.get('$type') == 'app.bsky.richtext.facet#tag':
                            tags.append(feature.get('tag', '').lower())

                if tag.lower() in tags:
                    count += 1

            except json.JSONDecodeError:
                continue  # skip malformed l
        return count

print(count_trump_tagged_posts('trump'))


2094


In [145]:
import json
from datetime import datetime
from collections import Counter

def get_post_days(file_path=file, with_counts=False):
    dates = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                p = json.loads(line)
            except json.JSONDecodeError:
                continue

            created = (p.get("record") or {}).get("createdAt")
            if not created:
                continue

            # Safe ISO parsing (handles trailing 'Z')
            try:
                d = datetime.fromisoformat(created.replace("Z", "+00:00")).date().isoformat()
            except Exception:
                d = created[:10]  # fallback

            dates.append(d)

    if with_counts:
        return dict(sorted(Counter(dates).items()))  # {date: num_posts}
    return sorted(set(dates))  # ["YYYY-MM-DD", ...]
get_post_days()

['2025-10-08',
 '2025-10-09',
 '2025-10-11',
 '2025-10-12',
 '2025-10-13',
 '2025-10-14',
 '2025-10-15',
 '2025-10-16',
 '2025-10-17',
 '2025-10-18',
 '2025-10-19',
 '2025-10-20',
 '2025-10-21',
 '2025-10-22']